In [1]:
# in Step 1, we query the original knowledge graph in SAP HANA to extract all triples related to Ukraine

import os
from dotenv import load_dotenv
load_dotenv()
from hdbcli import dbapi

# HANA Credentials
address = os.getenv("HANA_HOST")
port = os.getenv("HANA_PORT")
user = os.getenv("HANA_USER")
password = os.getenv("HANA_PASSWORD")

connection = dbapi.connect(address=address, port=port, user=user, password=password)
cursor = connection.cursor()

graph_query = """
SELECT * FROM SPARQL_TABLE('
  SELECT ?s ?p ?o 
  WHERE { GRAPH <ukraine_knowledgegraph> { ?s ?p ?o . } }
');
"""

cursor.execute(graph_query)
graph_rows = cursor.fetchall()

cursor.close()
connection.close()

import pandas as pd
graph_df = pd.DataFrame(graph_rows, columns=['subject', 'predicate', 'object'])

graph_df.to_dict(orient='records')

,subject,predicate,object
0,C1,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,Country
1,C2,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,Country
2,C1,name,Ukraine
3,C2,name,Russia


In [6]:
str(graph_rows)

"[('C1', 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'Country'), ('C2', 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'Country'), ('C1', 'name', 'Ukraine'), ('C2', 'name', 'Russia')]"

In [9]:
# In step 2, we get the summary of the latest developments regarding the war in Ukraine

import requests
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# First API call: Initialize data
initialize_url = 'https://geopolitics_api.cfapps.ap10.hana.ondemand.com/initialize-data/'
headers = {'accept': 'application/json'}

initialize_response = requests.get(initialize_url, headers=headers, verify=False)
print(f"Initialize status: {initialize_response.status_code}")

# Second API call: Get geopolitics prompt
prompt_url = 'https://geopolitics_api.cfapps.ap10.hana.ondemand.com/geopolitics_prompt/'
headers = {
    'accept': 'application/json',
    'Content-Type': 'application/json'
}
data = {'prompt': ''}

prompt_response = requests.post(prompt_url, headers=headers, json=data, verify=False)
summary_output = prompt_response.text

summary_output

Initialize status: 200


'"Bottom line\\n- Russia is pressing along the Donetsk axis toward Pokrovsk with localized gains and heavy attrition, while Ukraine’s long‑range strikes continue to degrade Russian logistics and fuel infrastructure. Winter weather is slowing tempo but favoring small‑unit infiltration and drone‑centric warfare. Moscow is amplifying a “long‑war, inevitable victory” narrative to shape prospective talks on its terms; negotiation chatter is rising in Europe and the US, but Russian officials still signal no willingness to compromise.\\n\\nOperational picture\\n- Donetsk (Pokrovsk sector): Confirmed Russian advances in Rodynske and along the T‑0504 near Rivne; reports indicate Ukrainian withdrawals from Lysivka and Sukhyi Yar. Russian infiltrations persist toward Myrnohrad and Pokrovsk. Russia claims it “liberated” Pokrovsk and is fighting in Myrnohrad; this is unconfirmed and likely exaggerated. Expect urban fighting risk and continued attritional assaults.\\n- Lyman–Siversk: High‑intensity 

In [2]:
# Package everything as functions

import os
from dotenv import load_dotenv
load_dotenv()
from hdbcli import dbapi

def get_ua_knowledgegraph():
    """Get the knowledge graph related to Ukraine from SAP HANA."""
    # HANA Credentials
    address = os.getenv("HANA_HOST")
    port = os.getenv("HANA_PORT")
    user = os.getenv("HANA_USER")
    password = os.getenv("HANA_PASSWORD")

    connection = dbapi.connect(address=address, port=port, user=user, password=password)
    cursor = connection.cursor()

    graph_query = """
    SELECT * FROM SPARQL_TABLE('
    SELECT ?s ?p ?o 
    WHERE { GRAPH <ukraine_knowledgegraph> { ?s ?p ?o . } }
    ');
    """

    cursor.execute(graph_query)
    graph_rows = cursor.fetchall()

    cursor.close()
    connection.close()

    return graph_rows

import requests
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def get_latest_summary():

    # First API call: Initialize data
    initialize_url = 'https://geopolitics_api.cfapps.ap10.hana.ondemand.com/initialize-data/'
    headers = {'accept': 'application/json'}

    initialize_response = requests.get(initialize_url, headers=headers, verify=False)
    print(f"Initialize status: {initialize_response.status_code}")

    # Second API call: Get geopolitics prompt
    prompt_url = 'https://geopolitics_api.cfapps.ap10.hana.ondemand.com/geopolitics_prompt/'
    headers = {
        'accept': 'application/json',
        'Content-Type': 'application/json'
    }
    data = {'prompt': ''}

    prompt_response = requests.post(prompt_url, headers=headers, json=data, verify=False)
    summary_output = prompt_response.text

    return summary_output

In [3]:
# In step 3, we use GPT-5 to generate the SPARQL notation to update the knowledge graph

from datetime import date

AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL")
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP")
os.environ['AICORE_CLIENT_ID'] = AICORE_CLIENT_ID
os.environ['AICORE_CLIENT_SECRET'] = AICORE_CLIENT_SECRET
os.environ['AICORE_AUTH_URL'] = AICORE_AUTH_URL
os.environ['AICORE_BASE_URL'] = AICORE_BASE_URL
os.environ['AICORE_RESOURCE_GROUP'] = AICORE_RESOURCE_GROUP

from gen_ai_hub.proxy.native.openai import chat

def sparql_prompt(summary, graph):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"You are a geopolitical analyst and an ontology expert proficient in SPARQL notation tasked with extracting subject-predicate-object (SPO) triples from documents about the war in Ukraine." +
                            f"Here is the document for processing regarding the war in Ukraine: {summary}" +
                            f"Here is the existing knowledge graph about the war in Ukraine: {str(graph)}" +
                            f"Find all the SPO triples resulting from the document and draft a SPARQL notation that adds all the new SPO triples to the knowledge graph." +
                            f"Return ONLY the pure SPARQL INSERT DATA statement, like this example:" +
                            ' INSERT DATA { GRAPH <ukraine_knowledgegraph> { <C1> a <Country>; <name> "Ukraine". <C2> a <Country>; <name> "Russia". }} ' +
                            f"CRITICAL: Use regular double quotes for string literals in SPARQL. Do NOT use backslashes. Do NOT escape quotes with backslash." +
                            f"Return ONLY the SPARQL INSERT DATA statement on a single line. No wrapper, no backslashes, no line breaks." +
                            f"One more thing: For SPO triples that might be temporally sensitive, ensure to include appropriate temporal qualifiers (e.g., today's date, which is: {str(date.today())})."
                }
            ]
        }
    ]
    return messages

def write_prompt_messages(messages, model_name = "gpt-5"): # Models can be changed here
    """Send messages to the model and return the response."""
    kwargs = dict(model_name=model_name, messages=messages)
    response = chat.completions.create(**kwargs)
    return response.to_dict()["choices"][0]["message"]["content"]

def get_sparql(summary, graph):
    """Get response from GPT 5."""
    messages = sparql_prompt(summary, graph)
    answer_entities = write_prompt_messages(messages)
    return answer_entities

In [ ]:
# In step 4, we update the knowledge graph in SAP HANA with the generated SPARQL notation

def update_knowledge_graph(sparql_notation):
    """Update the knowledge graph in SAP HANA."""
    # HANA Credentials
    address = os.getenv("HANA_HOST")
    port = os.getenv("HANA_PORT")
    user = os.getenv("HANA_USER")
    password = os.getenv("HANA_PASSWORD")

    connection = dbapi.connect(address=address, port=port, user=user, password=password)
    cursor = connection.cursor()

    try:
        # Clean up the SPARQL notation if needed
        if "CALL SPARQL_EXECUTE" in sparql_notation:
            # Extract just the SPARQL INSERT statement from inside the CALL
            start = sparql_notation.find("'") + 1
            end = sparql_notation.rfind("'", 0, sparql_notation.rfind("'"))
            sparql_insert = sparql_notation[start:end]
            
            # Escape single quotes for SQL by doubling them
            sparql_insert_escaped = sparql_insert.replace("'", "''")
            
            # Use SPARQL_EXECUTE with proper parameters
            query = f"CALL SPARQL_EXECUTE('{sparql_insert_escaped}', '', ?, ?)"
            cursor.execute(query)
        else:
            # If it's already a clean SPARQL statement, execute directly
            cursor.execute(sparql_notation)
        
        connection.commit()
        print("Knowledge graph updated successfully!")
        
    except Exception as e:
        print(f"Error updating knowledge graph: {e}")
        print(f"SPARQL statement preview: {sparql_notation[:500]}...")
        connection.rollback()
    finally:
        cursor.close()
        connection.close()

Error updating knowledge graph: (129, "transaction rolled back by an internal error: generic. Error - at line 1 column 85 near '\\': Syntax error - syntax error")
SPARQL statement preview: "CALL SPARQL_EXECUTE(' INSERT DATA { GRAPH <ukraine_knowledgegraph> { <AxisPokrovsk> a <FrontArc>; <name> \"Pokrovsk axis\"; <locatedIn> <RegionDonetsk>. <RegionDonetsk> a <Region>; <name> \"Donetsk Oblast\"; <locatedIn> <C1>. <L19> a <Settlement>; <name> \"Starytsia\"; <locatedIn> <C1>. <L20> a <Settlement>; <name> \"Hulyaipole\"; <locatedIn> <C1>. <L21> a <City>; <name> \"Cheboksary\"; <locatedIn> <C2>. <L24> a <City>; <name> \"Kupyansk\"; <locatedIn> <C1>. <L25> a <Settlement>; <name> \"Borov...


In [4]:
# Test the functions

graph_rows = get_ua_knowledgegraph()
summary_output = get_latest_summary()
sparql_notation = get_sparql(summary_output, graph_rows)
print(sparql_notation)

Initialize status: 200
INSERT DATA { GRAPH <ukraine_knowledgegraph> { <Kostyantynivka> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> "Location"; <name> "Kostyantynivka". <Druzhkivka> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> "Location"; <name> "Druzhkivka". <Stepnohirsk> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> "Location"; <name> "Stepnohirsk". <Hulyaipole> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> "Location"; <name> "Hulyaipole". <Stepove> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> "Location"; <name> "Stepove". <Oleksandrivka> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> "Location"; <name> "Oleksandrivka". <Dnipropetrovsk> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> "Location"; <name> "Dnipropetrovsk". <Cheboksary> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> "Location"; <name> "Cheboksary". <Luhansk> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> "Location"; <name> "Luhansk". <Donetsk> <http://www.w3.org/1999/02/22-rdf-syntax-ns#

In [8]:
# In step 4, we update the knowledge graph in SAP HANA with the generated SPARQL notation

def update_knowledge_graph(sparql_notation):
    """Update the knowledge graph in SAP HANA."""
    # HANA Credentials
    address = os.getenv("HANA_HOST")
    port = os.getenv("HANA_PORT")
    user = os.getenv("HANA_USER")
    password = os.getenv("HANA_PASSWORD")

    connection = dbapi.connect(address=address, port=port, user=user, password=password)
    cursor = connection.cursor()

    try:
        # First, drop the existing graph
        drop_query = """
                    CALL SPARQL_EXECUTE('
                    DROP GRAPH <ukraine_knowledgegraph> 
                    ', '', ?, ?);
                    """
        cursor.execute(drop_query)
        connection.commit()
        print("Existing graph dropped")
        
        # Extract clean SPARQL if wrapped in CALL
        if "CALL SPARQL_EXECUTE" in sparql_notation:
            start = sparql_notation.find("'") + 1
            end = sparql_notation.rfind("'", 0, sparql_notation.rfind("'"))
            sparql_insert = sparql_notation[start:end]
        else:
            sparql_insert = sparql_notation.strip()
        
        # Clean up any backslash escaping from GPT
        sparql_insert = sparql_insert.replace('\\"', '"').replace('\\', '')
        
        print(f"Executing SPARQL ({len(sparql_insert)} chars)")
        
        # Use the exact format that works
        query = f"""
                CALL SPARQL_EXECUTE(
                    '{sparql_insert.replace("'", "''")}', 
                    '', 
                    ?, 
                    ?
                );
                """
        cursor.execute(query)
        connection.commit()
        print("Knowledge graph updated successfully!")
        
    except Exception as e:
        print(f"Execution failed: {e}")
        connection.rollback()
    finally:
        cursor.close()
        connection.close()

update_knowledge_graph(sparql_notation)

Existing graph dropped
Executing SPARQL (11333 chars)
Knowledge graph updated successfully!
Knowledge graph updated successfully!


---

## LLM Integration

In [35]:
# in Step 1, we query the original knowledge graph in SAP HANA to extract all triples related to Ukraine (replicating earlier steps)

import os
from dotenv import load_dotenv
load_dotenv()
from hdbcli import dbapi

# HANA Credentials
address = os.getenv("HANA_HOST")
port = os.getenv("HANA_PORT")
user = os.getenv("HANA_USER")
password = os.getenv("HANA_PASSWORD")

connection = dbapi.connect(address=address, port=port, user=user, password=password)
cursor = connection.cursor()

graph_query = """
SELECT * FROM SPARQL_TABLE('
  SELECT ?s ?p ?o 
  WHERE { GRAPH <ukraine_knowledgegraph> { ?s ?p ?o . } }
');
"""

cursor.execute(graph_query)
graph_rows = cursor.fetchall()

cursor.close()
connection.close()

import pandas as pd
graph_df = pd.DataFrame(graph_rows, columns=['subject', 'predicate', 'object'])
graph_df.head()

,subject,predicate,object
0,C1,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,Country
1,C2,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,Country
2,C1,name,Ukraine
3,C2,name,Russia
4,Force_Multinational_Peacekeeping_Ukraine,name,Multinational peacekeeping force for Ukraine


In [36]:
from langchain_hana import HanaRdfGraph

connection = dbapi.connect(address=address, port=port, user=user, password=password)

graph_uri = "ukraine_knowledgegraph"
graph = HanaRdfGraph(
    graph_uri=graph_uri,
    connection = connection,
    auto_extract_ontology = True
)

print(graph.get_schema.serialize(format='turtle'))

@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<file:///c:/Users/I745534/OneDrive%20-%20SAP%20SE/Documents/Daedalus%20Alpha/DEIG/Hackathon/3%20Oslo/ai_hackathon_oslo/ukraine/estimates> a owl:ObjectProperty ;
    rdfs:label "estimates" ;
    rdfs:domain <file:///c:/Users/I745534/OneDrive%20-%20SAP%20SE/Documents/Daedalus%20Alpha/DEIG/Hackathon/3%20Oslo/ai_hackathon_oslo/ukraine/Country> ;
    rdfs:range xsd:uri .

<file:///c:/Users/I745534/OneDrive%20-%20SAP%20SE/Documents/Daedalus%20Alpha/DEIG/Hackathon/3%20Oslo/ai_hackathon_oslo/ukraine/name> a owl:DatatypeProperty ;
    rdfs:label "name" ;
    rdfs:domain <file:///c:/Users/I745534/OneDrive%20-%20SAP%20SE/Documents/Daedalus%20Alpha/DEIG/Hackathon/3%20Oslo/ai_hackathon_oslo/ukraine/Country> ;
    rdfs:range xsd:string .

<file:///c:/Users/I745534/OneDrive%20-%20SAP%20SE/Documents/Daedalus%20Alpha/DEIG/Hackathon/3%20Oslo/ai_hacka

In [37]:
from langchain_hana import HanaSparqlQAChain
from gen_ai_hub.proxy.langchain.init_models import init_llm

model_to_use = "gpt-5"
llm = init_llm(model_name=model_to_use)

from langchain_core.prompts import PromptTemplate

SPARQL_GENERATION_SELECT_TEMPLATE = PromptTemplate(
    input_variables=["schema", "prompt"], 
    template="""You are an expert in SPARQL queries. Given the following RDF schema and a user question, generate a SPARQL SELECT query to retrieve relevant information.

Schema:
{schema}

User Question: {prompt}

Generate ONLY a SPARQL SELECT query (no explanations, no markdown). The query must include a WHERE clause.
Example format:
SELECT ?subject ?predicate ?object
WHERE {{
    ?subject ?predicate ?object .
    FILTER(...)
}}
"""
)

SPARQL_QA_TEMPLATE = PromptTemplate(
    input_variables=["context", "prompt"], 
    template="""Based on the following information from a knowledge graph, provide a comprehensive answer to the user's question.

User Question: {prompt}

Knowledge Graph Data:
{context}

Provide a clear, detailed summary based on this data. If the data is empty or doesn't contain relevant information, state that clearly.
"""
)

In [38]:
from typing import cast

chain = HanaSparqlQAChain.from_llm(
    llm=llm,
    verbose=True,
    allow_dangerous_requests=True,
    graph=graph,
    sparql_generation_prompt=cast(PromptTemplate, SPARQL_GENERATION_SELECT_TEMPLATE),
    qa_prompt=cast(PromptTemplate, SPARQL_QA_TEMPLATE)
)

chain.invoke({"query": "Give me all triples related to Ukraine."})



> Entering new HanaSparqlQAChain chain...


AttributeError: 'RunnableSequence' object has no attribute 'output_key'